# 04 — Churn Prediction (Leakage Audit + Modelling)

**Owner:** Manchidi Magaseng John (Machine Learning)

Mandatory leakage audit, then two modelling experiments: A (all features) and B (leakage-resistant, primary).

In [1]:
# Make the src package importable whether or not it is pip-installed.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(ROOT)
print('project root:', ROOT)

project root: C:\Users\johnm\Big_Data_Challenge_Project


In [2]:
import pandas as pd
from churn_project import feature_engineering as fe, model_training as mt, model_evaluation as me
from churn_project.config import CONFIG
df = pd.read_csv(CONFIG.path('clean_csv'))

## Leakage audit

In [3]:
audit = fe.audit_leakage(df)
import json; print(json.dumps(audit, indent=2, default=str))

{
  "days_since_threshold_reconstruction": {
    "threshold": 90,
    "accuracy": 1.0
  },
  "days_since_by_class": {
    "0": {
      "min": 1,
      "max": 90,
      "mean": 8.786
    },
    "1": {
      "min": 91,
      "max": 719,
      "mean": 300.617
    }
  },
  "target_correlations": {
    "days_since_last_purchase": 0.8358,
    "account_age_days": -0.1199,
    "age": -0.0088,
    "total_orders": -0.0142,
    "total_spent_usd": -0.0088,
    "avg_order_value": -0.0055,
    "is_premium_member": 0.0131
  },
  "verdict": {
    "days_since_last_purchase": "TARGET-DEFINING LEAKAGE",
    "date_columns": "LEAKAGE (encode the same recency signal as days_since)",
    "note": "Churn is a snapshot rule, not a future event. Removing recency/date fields gives a leakage-resistant but weaker and more honest model."
  }
}


## Two experiments: fit and evaluate

In [4]:
rows = []
for exp in ('A', 'B'):
    X, y, feats = fe.build_xy(df, exp)
    Xtr, Xte, ytr, yte = mt.split_data(X, y)
    for name, pipe in mt.build_models(feats['numeric'], feats['categorical']).items():
        pipe.fit(Xtr, ytr)
        r = me.evaluate_model(name, pipe, Xte, yte); r['experiment'] = exp
        rows.append(r)
comp = pd.DataFrame(rows).sort_values(['experiment','f1'], ascending=[True, False])
comp

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,experiment
2,RandomForest,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,A
1,LogisticRegression,0.9945,0.9869,0.9987,0.9927,0.9999,0.9998,A
0,DummyClassifier,0.6230,0.0000,0.0000,0.0000,0.5000,0.3770,A
4,LogisticRegression,0.5200,0.4019,0.5597,0.4678,0.5506,0.4078,B
5,RandomForest,0.5615,0.4075,0.3594,0.3820,0.5353,0.4105,B
3,DummyClassifier,0.6230,0.0000,0.0000,0.0000,0.5000,0.3770,B


**Findings:** Experiment A ~100% (leakage). Experiment B is near the majority baseline (ROC-AUC ~0.54), so no non-recency behaviour predicts churn. The leakage-resistant model is the honest, primary result.

## Full orchestration (optional)
Run the whole project end-to-end and persist all artefacts:

In [5]:
# from churn_project.run_all import main
# summary = main(use_spark=True)
# summary